# Multimodal Housing Price Prediction
Predict house prices using both **tabular features** and **house images**.

In [ ]:
import os, json, random, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
import joblib

from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

In [ ]:
# Config
MAX_ROWS = None
BATCH_SIZE = 32
NUM_WORKERS = 2
EPOCHS = 5
LR = 1e-3
PATIENCE = 2
IMAGE_SIZE = 224
TAB_HIDDEN = 128
FUSION_HIDDEN = 128

OUT = Path("/kaggle/working/multimodal_house_price")
OUT.mkdir(parents=True, exist_ok=True)
print(OUT)

In [ ]:
# Auto-discover dataset
def find_csv(root="/kaggle/input"):
    cands = list(Path(root).rglob("*.csv"))
    if not cands:
        raise FileNotFoundError("No CSV found")
    score_keys = ["socal", "house", "price", "housing", "image"]
    scored = []
    for p in cands:
        name = p.name.lower()
        score = sum(k in name for k in score_keys)
        scored.append((score, str(p)))
    scored.sort(key=lambda x: (-x[0], x[1]))
    return Path(scored[0][1])

def find_img_root(root="/kaggle/input"):
    dirs = [p for p in Path(root).rglob("*") if p.is_dir()]
    score_keys = ["pics", "images", "image", "socal", "house"]
    scored = []
    for p in dirs:
        name = p.name.lower()
        score = sum(k in name for k in score_keys)
        scored.append((score, str(p)))
    scored.sort(key=lambda x: (-x[0], x[1]))
    return Path(scored[0][1])

CSV_PATH = find_csv()
IMG_ROOT = find_img_root()
print(CSV_PATH)
print(IMG_ROOT)

In [ ]:
df = pd.read_csv(CSV_PATH)
df.columns = [c.strip() for c in df.columns]
print(df.shape)
display(df.head())

In [ ]:
def detect_target(cols):
    for c in cols:
        lc = c.lower()
        if lc in {"price", "target", "saleprice", "soldprice"} or "price" in lc:
            return c
    for c in cols:
        if pd.api.types.is_numeric_dtype(df[c]) and df[c].nunique() > 100 and c.lower() not in {"id", "image_id"}:
            return c
    raise ValueError("Target not found")

def detect_image_col(cols):
    for c in cols:
        lc = c.lower()
        if lc in {"image_id", "image", "img", "filename", "path"} or "image" in lc:
            return c
    return None

TARGET = detect_target(df.columns)
IMAGE_COL = detect_image_col(df.columns)
print("TARGET:", TARGET)
print("IMAGE_COL:", IMAGE_COL)

In [ ]:
EXTS = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
def resolve_img(v):
    if pd.isna(v):
        return None
    raw = str(v).strip()
    p = Path(raw)
    if p.exists():
        return p
    cands = [IMG_ROOT / raw]
    if p.suffix.lower() in EXTS:
        cands.append(IMG_ROOT / p.name)
    else:
        cands += [IMG_ROOT / f"{raw}{ext}" for ext in EXTS]
    for c in cands:
        if c.exists():
            return c
    stem = p.stem if p.suffix else raw
    for ext in EXTS:
        m = list(IMG_ROOT.rglob(f"{stem}{ext}"))
        if m:
            return m[0]
    return None

df["image_path"] = df[IMAGE_COL].apply(resolve_img) if IMAGE_COL else None
df = df.dropna(subset=[TARGET]).copy()
df = df[df["image_path"].notna()].copy()
if MAX_ROWS and len(df) > MAX_ROWS:
    df = df.sample(MAX_ROWS, random_state=SEED)
df = df.reset_index(drop=True)
print(df.shape)

In [ ]:
# EDA
display(df.describe(include="all").T)
plt.figure(figsize=(10,4))
sns.histplot(df[TARGET], bins=40, kde=True)
plt.title("Price Distribution")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10,4))
sns.histplot(np.log1p(df[TARGET]), bins=40, kde=True)
plt.title("Log Price Distribution")
plt.tight_layout()
plt.show()

num_cols = [c for c in df.columns if c not in {TARGET, IMAGE_COL, "image_path"} and pd.api.types.is_numeric_dtype(df[c])]
cat_cols = [c for c in df.columns if c not in {TARGET, IMAGE_COL, "image_path"} and not pd.api.types.is_numeric_dtype(df[c])]
print("Numerical:", num_cols)
print("Categorical:", cat_cols)

for col in cat_cols[:3]:
    plt.figure(figsize=(10,3))
    sns.barplot(x=df[col].astype(str).value_counts().head(10).index,
                y=df[col].astype(str).value_counts().head(10).values)
    plt.xticks(rotation=30, ha="right")
    plt.title(col)
    plt.tight_layout()
    plt.show()

In [ ]:
# Sample images
fig, axes = plt.subplots(3,3, figsize=(12,12))
for ax, (_, row) in zip(axes.flatten(), df.sample(min(9, len(df)), random_state=SEED).iterrows()):
    ax.imshow(Image.open(row["image_path"]).convert("RGB"))
    ax.set_title(f"{row[TARGET]:,.0f}", fontsize=9)
    ax.axis("off")
for ax in axes.flatten()[len(df.sample(min(9, len(df)), random_state=SEED)):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=SEED)
print(train_df.shape, val_df.shape, test_df.shape)

In [ ]:
# Tabular preprocessing
num_cols = [c for c in num_cols if c != TARGET]
try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", ohe)]), cat_cols),
], remainder="drop")

Xtr = pre.fit_transform(train_df[num_cols + cat_cols])
Xva = pre.transform(val_df[num_cols + cat_cols])
Xte = pre.transform(test_df[num_cols + cat_cols])

ytr = np.log1p(train_df[TARGET].values.astype(np.float32))
yva = np.log1p(val_df[TARGET].values.astype(np.float32))
yte = np.log1p(test_df[TARGET].values.astype(np.float32))

tab_dim = Xtr.shape[1]
print("tab_dim:", tab_dim)

In [ ]:
# Image transforms and dataset
tr_tfms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(8),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
ev_tfms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

class HouseDS(Dataset):
    def __init__(self, frame, tab, y, tfm):
        self.f = frame.reset_index(drop=True)
        self.tab = torch.tensor(tab, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.tfm = tfm
    def __len__(self): return len(self.f)
    def __getitem__(self, i):
        r = self.f.iloc[i]
        img = self.tfm(Image.open(r["image_path"]).convert("RGB"))
        return img, self.tab[i], self.y[i]

tr_ds = HouseDS(train_df, Xtr, ytr, tr_tfms)
va_ds = HouseDS(val_df, Xva, yva, ev_tfms)
te_ds = HouseDS(test_df, Xte, yte, ev_tfms)

tr_ld = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
va_ld = DataLoader(va_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
te_ld = DataLoader(te_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

In [ ]:
# Models
class TabNet(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, TAB_HIDDEN), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(TAB_HIDDEN, TAB_HIDDEN//2), nn.ReLU(),
            nn.Linear(TAB_HIDDEN//2, 1)
        )
    def forward(self, tab): return self.net(tab).squeeze(-1)

class ImgEnc(nn.Module):
    def __init__(self):
        super().__init__()
        b = resnet18(weights=ResNet18_Weights.DEFAULT)
        self.f = nn.Sequential(*list(b.children())[:-1])
        self.d = b.fc.in_features
    def forward(self, x): return self.f(x).view(x.size(0), -1)

class ImgNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = ImgEnc()
        self.head = nn.Sequential(nn.Linear(self.enc.d, 128), nn.ReLU(), nn.Dropout(0.2), nn.Linear(128,1))
    def forward(self, img): return self.head(self.enc(img)).squeeze(-1)

class MultiNet(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.enc = ImgEnc()
        self.tab = nn.Sequential(nn.Linear(d, TAB_HIDDEN), nn.ReLU(), nn.Dropout(0.2), nn.Linear(TAB_HIDDEN, TAB_HIDDEN//2), nn.ReLU())
        self.head = nn.Sequential(nn.Linear(self.enc.d + TAB_HIDDEN//2, FUSION_HIDDEN), nn.ReLU(), nn.Dropout(0.2), nn.Linear(FUSION_HIDDEN, 1))
    def forward(self, img, tab):
        return self.head(torch.cat([self.enc(img), self.tab(tab)], dim=1)).squeeze(-1)

def freeze_img(m):
    for p in m.enc.parameters():
        p.requires_grad = False

In [ ]:
# Train / eval helpers
def metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred, squared=False),
    }

def inverse(x): return np.expm1(x)

def run_epoch(model, loader, opt=None, mode="multi"):
    train = opt is not None
    model.train(train)
    crit = nn.SmoothL1Loss()
    tot = 0.0
    preds, trues = [], []
    for img, tab, y in loader:
        img, tab, y = img.to(DEVICE), tab.to(DEVICE), y.to(DEVICE)
        if train: opt.zero_grad()
        out = model(tab) if mode=="tab" else model(img) if mode=="img" else model(img, tab)
        loss = crit(out, y)
        if train:
            loss.backward()
            opt.step()
        tot += loss.item() * len(y)
        preds.append(out.detach().cpu().numpy()); trues.append(y.detach().cpu().numpy())
    return tot / len(loader.dataset), np.concatenate(preds), np.concatenate(trues)

def train_model(model, tr_ld, va_ld, mode="multi"):
    model = model.to(DEVICE)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", patience=1, factor=0.5)
    best, best_state, bad = float("inf"), None, 0
    hist = []
    for ep in range(1, EPOCHS + 1):
        tr_loss, _, _ = run_epoch(model, tr_ld, opt, mode)
        va_loss, _, _ = run_epoch(model, va_ld, None, mode)
        hist.append((ep, tr_loss, va_loss))
        print(f"Epoch {ep}: train {tr_loss:.4f} | val {va_loss:.4f}")
        sched.step(va_loss)
        if va_loss < best:
            best, bad = va_loss, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                print("Early stopping")
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, pd.DataFrame(hist, columns=["epoch", "train_loss", "val_loss"])

def eval_model(model, loader, mode="multi"):
    _, p_log, y_log = run_epoch(model, loader, None, mode)
    p, y = inverse(p_log), inverse(y_log)
    return metrics(y, p), p, y, p_log, y_log

In [ ]:
# Baseline: tabular only
tab_model = TabNet(tab_dim)
tab_model, tab_hist = train_model(tab_model, tr_ld, va_ld, mode="tab")

In [ ]:
# Multimodal model
mm_model = MultiNet(tab_dim)
freeze_img(mm_model)
mm_model, mm_hist = train_model(mm_model, tr_ld, va_ld, mode="multi")

In [ ]:
# Evaluate
tab_val_m, _, _, _, _ = eval_model(tab_model, va_ld, mode="tab")
tab_test_m, _, _, _, _ = eval_model(tab_model, te_ld, mode="tab")

mm_val_m, mm_pred, mm_true, _, _ = eval_model(mm_model, te_ld, mode="multi")

res = pd.DataFrame([
    {"Model":"Tabular Only", **tab_test_m},
    {"Model":"Multimodal Fusion", **mm_val_m},
])
display(res)

plt.figure(figsize=(10,4))
plt.plot(tab_hist["epoch"], tab_hist["train_loss"], label="Tab train")
plt.plot(tab_hist["epoch"], tab_hist["val_loss"], label="Tab val")
plt.plot(mm_hist["epoch"], mm_hist["train_loss"], label="MM train")
plt.plot(mm_hist["epoch"], mm_hist["val_loss"], label="MM val")
plt.legend(); plt.title("Training Curves"); plt.tight_layout(); plt.show()

plt.figure(figsize=(6,6))
plt.scatter(mm_true, mm_pred, alpha=0.35)
mn, mx = min(mm_true.min(), mm_pred.min()), max(mm_true.max(), mm_pred.max())
plt.plot([mn,mx],[mn,mx],"r--")
plt.xlabel("Actual"); plt.ylabel("Predicted"); plt.title("Multimodal: Actual vs Predicted")
plt.tight_layout(); plt.show()

In [ ]:
# Save artifacts
art = OUT / "artifacts"
art.mkdir(parents=True, exist_ok=True)
torch.save(mm_model.state_dict(), art / "multimodal_model.pt")
torch.save(tab_model.state_dict(), art / "tabular_model.pt")
joblib.dump(pre, art / "tab_preprocessor.joblib")
with open(art / "metadata.json", "w") as f:
    json.dump({
        "csv_path": str(CSV_PATH),
        "image_root": str(IMG_ROOT),
        "target": TARGET,
        "image_col": IMAGE_COL,
        "num_cols": num_cols,
        "cat_cols": cat_cols,
        "tab_dim": tab_dim
    }, f, indent=2)
print("Saved:", art)

In [ ]:
# Streamlit app
streamlit_code = r'''
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import streamlit as st
import torch
import torch.nn as nn
from PIL import Image
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights

BASE = Path(__file__).parent / "artifacts"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

with open(BASE / "metadata.json") as f:
    META = json.load(f)

df = pd.read_csv(META["csv_path"])
df.columns = [c.strip() for c in df.columns]

TARGET = META["target"]
IMAGE_COL = META["image_col"]
IMG_ROOT = Path(META["image_root"])
NUM_COLS = META["num_cols"]
CAT_COLS = META["cat_cols"]

EXTS = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
def resolve_img(v):
    raw = str(v).strip()
    p = Path(raw)
    if p.exists(): return p
    for ext in EXTS:
        c = IMG_ROOT / f"{raw}{ext}"
        if c.exists(): return c
    c = IMG_ROOT / raw
    if c.exists(): return c
    return None

df["image_path"] = df[IMAGE_COL].apply(resolve_img)
df = df.dropna(subset=[TARGET, "image_path"]).reset_index(drop=True)

pre = joblib.load(BASE / "tab_preprocessor.joblib")
tab_dim = pre.transform(df[NUM_COLS + CAT_COLS].head(1)).shape[1]

class ImgEnc(nn.Module):
    def __init__(self):
        super().__init__()
        b = resnet18(weights=ResNet18_Weights.DEFAULT)
        self.f = nn.Sequential(*list(b.children())[:-1])
        self.d = b.fc.in_features
    def forward(self, x): return self.f(x).view(x.size(0), -1)

class MultiNet(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.enc = ImgEnc()
        self.tab = nn.Sequential(nn.Linear(d, 128), nn.ReLU(), nn.Linear(128, 64), nn.ReLU())
        self.head = nn.Sequential(nn.Linear(self.enc.d + 64, 128), nn.ReLU(), nn.Linear(128, 1))
    def forward(self, img, tab):
        return self.head(torch.cat([self.enc(img), self.tab(tab)], dim=1)).squeeze(-1)

model = MultiNet(tab_dim).to(DEVICE)
model.load_state_dict(torch.load(BASE / "multimodal_model.pt", map_location=DEVICE))
model.eval()

tfm = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

st.title("Multimodal House Price Demo")
idx = st.number_input("Row index", 0, len(df)-1, 0, 1)
row = df.iloc[int(idx)]
st.image(Image.open(row["image_path"]).convert("RGB"), use_container_width=True)

tab_x = pre.transform(row[NUM_COLS + CAT_COLS].to_frame().T)
tab_x = torch.tensor(tab_x, dtype=torch.float32).to(DEVICE)
img = tfm(Image.open(row["image_path"]).convert("RGB")).unsqueeze(0).to(DEVICE)

with torch.no_grad():
    pred_log = model(img, tab_x).cpu().numpy()[0]
pred = float(np.expm1(pred_log))

st.metric("Predicted price", f"{pred:,.0f}")
st.metric("Actual price", f"{row[TARGET]:,.0f}")
'''
(OUT / "streamlit_app.py").write_text(streamlit_code, encoding="utf-8")
print("Streamlit app written.")

## Final Summary
- **Tabular-only baseline** trained successfully
- **Multimodal fusion** combines a CNN image encoder with tabular features
- The notebook reports **MAE** and **RMSE**
- A **Streamlit demo** is generated for deployment